In [1]:
%%writefile main.py
import math
import numpy as np
import os
import sys
import subprocess
from collections import defaultdict, namedtuple

# ============================================================
# 1. DYNAMIC DEPENDENCY INJECTION & MODEL LOADER
# ============================================================
# Safely find the agent directory (Handles Kaggle's exec() quirk)
try:
    AGENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # When Kaggle runs the agent as a string, __file__ is undefined.
    # Submitted tarballs are always extracted here:
    AGENT_DIR = "/kaggle_simulations/agent"
    if not os.path.exists(AGENT_DIR):
        AGENT_DIR = os.getcwd()

ort = None
try:
    import onnxruntime as ort
except ImportError:
    print("Installing ONNX dependencies from local wheels...", file=sys.stderr)
    wheel_paths =[
        os.path.join(AGENT_DIR, "wheels_311"),
        "/kaggle_simulations/agent/wheels_311",
        "/kaggle/input/datasets/pjleek/euro-step-v4/wheels_311",
        "/kaggle/input/euro-step-v4/wheels_311"
    ]
    for wp in wheel_paths:
        if os.path.exists(wp):
            try:
                # Install directly into the local agent folder to avoid permissions issues
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", 
                    "--quiet", "--no-index", "--find-links=" + wp, 
                    "onnxruntime", "onnx"
                ])
                import onnxruntime as ort
                print(f"SUCCESS: Installed ONNX from {wp}", file=sys.stderr)
                break
            except Exception as e:
                print(f"Failed to install from {wp}: {e}", file=sys.stderr)

onnx_session = None
_logged_status = False

if ort:
    model_paths =[
        os.path.join(AGENT_DIR, "orbit_model.onnx"),
        "/kaggle_simulations/agent/orbit_model.onnx",
        "/kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx",
        "/kaggle/input/euro-step-v4/orbit_model.onnx",
        "/kaggle/working/orbit_model.onnx"
    ]
    for mp in model_paths:
        if os.path.exists(mp):
            try:
                onnx_session = ort.InferenceSession(mp)
                print(f"SUCCESS: Loaded ONNX Brain from {mp}", file=sys.stderr)
                break
            except Exception as e:
                print(f"Error loading ONNX from {mp}: {e}", file=sys.stderr)

# ============================================================
# 2. Engine Configuration
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5
OVERCOMMIT_RATIO = 1.1
HORIZON = 100
TOTAL_STEPS = 500

# With BCELoss, values separate cleanly. 0.55 is a solid threshold.
MIN_NN_SCORE = 0.55 

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])

# ============================================================
# 3. Core Physics Engine (Autopilot)
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

def get_target_pos(tgt, turns, ang_vel, initial_planets, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids", []):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): return c["paths"][idx][f_idx]
        return None
    init = initial_planets.get(tgt.id)
    if not init: return (tgt.x, tgt.y)
    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0: return (tgt.x, tgt.y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, initial_planets, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): # 5-Iteration Precision Solver
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: return None, 999
        tx, ty = pos

    angle = math.atan2(ty - src.y, tx - src.x)
    sx = src.x + math.cos(angle) * (src.radius + 0.1)
    sy = src.y + math.sin(angle) * (src.radius + 0.1)
    ex = sx + math.cos(angle) * (eta * speed)
    ey = sy + math.sin(angle) * (eta * speed)
    
    # Sun Safety Check
    if point_to_segment_dist(CENTER_X, CENTER_Y, sx, sy, ex, ey) <= SUN_R + SUN_SAFETY: 
        return None, 999
    return angle, int(math.ceil(eta))

# ============================================================
# 4. Main Agent Loop
# ============================================================
def agent(obs, config=None):
    global _logged_status
    if not _logged_status:
        if not onnx_session:
            print("WARNING: ONNX Model failed to load. Falling back to Heuristic.", file=sys.stderr)
        _logged_status = True

    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    planets =[Planet(*p) for p in (obs.get("planets", []) if is_dict else getattr(obs, "planets",[]))]
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    initial_planets = {Planet(*p).id: Planet(*p) for p in (obs.get("initial_planets",[]) if is_dict else getattr(obs, "initial_planets",[]))}
    comets = obs.get("comets", []) if is_dict else getattr(obs, "comets",[])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids", []))
    
    my_planets =[p for p in planets if p.owner == player]
    if not my_planets: return[]
    
    owners = set([p.owner for p in planets if p.owner != -1])
    is_4p = 1.0 if len(owners) > 2 else 0.0

    moves =[]
    budget = {p.id: p.ships for p in my_planets}
    evals =[]
    targeted = set()

    # Iterate through potential moves
    for src in my_planets:
        avail = budget[src.id]
        if avail < 10: continue

        for tgt in planets:
            if tgt.owner == player: continue
            
            # Autopilot: Physics & Cost Simulator
            rough_eta = math.hypot(tgt.x - src.x, tgt.y - src.y) / fleet_speed(avail)
            base_cost = tgt.ships + (tgt.production * rough_eta if tgt.owner != -1 else 0) + 1
            cost = int(math.ceil(base_cost * OVERCOMMIT_RATIO))
            
            if cost >= avail or cost < 1: continue
            
            angle, exact_eta = plan_flight(src, tgt, cost, ang_vel, initial_planets, comets, comet_ids)
            if angle is None or exact_eta > 45: continue 

            # Economic Feature Calculation
            in_quad = 1.0 if (1 if src.x > 50 else -1) == (1 if tgt.x > 50 else -1) and (1 if src.y > 50 else -1) == (1 if tgt.y > 50 else -1) else 0.0
            
            net_present_value = (tgt.production * max(0, 500 - step - exact_eta)) / 1000.0
            capital_risk = cost / max(1.0, float(src.ships))
            time_discount = exact_eta / 50.0
            raw_yield = tgt.production / 5.0
            ownership = 1.0 if tgt.owner != -1 else 0.5
            phase = step / 500.0
            
            if onnx_session:
                # The CEO: Neural Network Evaluation
                feature_vector = np.array([[
                    net_present_value, capital_risk, time_discount, 
                    raw_yield, ownership, in_quad, phase, is_4p
                ]], dtype=np.float32)
                score = float(onnx_session.run(None, {'input': feature_vector})[0][0][0])
            else:
                # Fallback: Hand-crafted NPV Heuristic
                score = 0.5 + (net_present_value / max(0.1, capital_risk)) * 0.1 - (time_discount * 0.2) + (in_quad * 0.1)

            if score >= MIN_NN_SCORE: 
                evals.append((score, src.id, tgt.id, cost, angle))

    # Execution Loop
    evals.sort(key=lambda x: x[0], reverse=True)
    
    for score, src_id, tgt_id, cost, angle in evals:
        if budget[src_id] >= cost and tgt_id not in targeted:
            moves.append([src_id, angle, cost])
            budget[src_id] -= cost
            targeted.add(tgt_id)

    return moves

Writing main.py
